In [1]:
import pandas as pd
import numpy as np

In [2]:
##transactions missing in the gateway report##
ledger_df=pd.read_csv("ledger.csv")
gateway_df=pd.read_csv("gateway_export.csv")

ledger_ids=set(ledger_df['transaction_id'])
gateway_ids=set(gateway_df['transaction_id'])

print(f"Unique IDs in Ledger: {len(ledger_ids)}")
print(f"Unique IDs in Gateway: {len(gateway_ids)}")

missing_in_gateway=ledger_ids-gateway_ids
print(f"\nTransactions in Ledger but NOT in Gateway: {len(missing_in_gateway)}")
print(f"Transaction IDs: {missing_in_gateway}")

missing_in_gateway_list=list(missing_in_gateway)
missing_in_gateway_df=ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway_list)].copy()

missing_in_gateway_df['issue_type']="MISSING_IN_GATEWAY"
print(f"\nTransaction IDs missing from Gateway report- ")
missing_in_gateway_df

Unique IDs in Ledger: 547
Unique IDs in Gateway: 530

Transactions in Ledger but NOT in Gateway: 27
Transaction IDs: {'TXN200001', 'TXN100288', 'TXN100036', 'TXN100005', 'TXN300030', 'TXN100313', 'TXN100188', 'TXN100212', 'TXN100074', 'TXN100330', 'TXN100000', 'TXN100259', 'TXN100471', 'TXN100488', 'TXN100395', 'TXN100379', 'TXN100056', 'TXN100250', 'TXN100134', 'TXN100324', 'TXN100351', 'TXN100130', 'TXN100475', 'TXN100044', 'TXN100360', 'TXN100092', 'TXN100390'}

Transaction IDs missing from Gateway report- 


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score,issue_type
0,TXN100000,350,16,2026-01-29 11:13:00,2999,Wallet,captured,85,MISSING_IN_GATEWAY
5,TXN100005,196,3,2026-01-06 22:06:00,99,UPI,captured,60,MISSING_IN_GATEWAY
36,TXN100036,321,16,2026-01-25 16:31:00,49,UPI,captured,35,MISSING_IN_GATEWAY
44,TXN100044,309,15,2026-01-24 23:34:00,99,Netbanking,captured,62,MISSING_IN_GATEWAY
56,TXN100056,132,24,2026-01-07 14:44:00,299,Wallet,captured,21,MISSING_IN_GATEWAY
74,TXN100074,244,32,2026-01-18 07:44:00,149,UPI,captured,57,MISSING_IN_GATEWAY
92,TXN100092,157,37,2026-01-28 18:27:00,499,Wallet,captured,79,MISSING_IN_GATEWAY
130,TXN100130,211,3,2026-01-13 07:54:00,149,UPI,chargeback,40,MISSING_IN_GATEWAY
134,TXN100134,74,27,2026-01-17 14:00:00,799,UPI,captured,83,MISSING_IN_GATEWAY
188,TXN100188,32,31,2026-01-15 20:01:00,299,UPI,captured,51,MISSING_IN_GATEWAY


In [3]:
##transactions missing in the ledger (extras in gateway)##
missing_in_ledger=gateway_ids-ledger_ids
print(f"\nTransactions in Gateway but NOT in Ledger: {len(missing_in_ledger)}")
print(f"Transaction IDs: {missing_in_ledger}")

missing_in_ledger_list=list(missing_in_ledger)
missing_in_ledger_df=gateway_df[gateway_df['transaction_id'].isin(missing_in_ledger_list)].copy()

missing_in_ledger_df['issue_type']="MISSING_IN_LEDGER"
print(f"\nTransaction IDs missing from Ledger- ")
missing_in_ledger_df


Transactions in Gateway but NOT in Ledger: 10
Transaction IDs: {'TXNX9008', 'TXNX9000', 'TXNX9001', 'TXNX9005', 'TXNX9007', 'TXNX9006', 'TXNX9009', 'TXNX9003', 'TXNX9004', 'TXNX9002'}

Transaction IDs missing from Ledger- 


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score,issue_type
520,TXNX9000,262,25,2026-01-14 00:00:00,149,Netbanking,captured,54,MISSING_IN_LEDGER
521,TXNX9001,96,32,2026-01-21 00:00:00,4999,Wallet,captured,71,MISSING_IN_LEDGER
522,TXNX9002,86,32,2026-01-10 00:00:00,149,Wallet,captured,40,MISSING_IN_LEDGER
523,TXNX9003,231,40,2026-01-02 00:00:00,799,UPI,captured,62,MISSING_IN_LEDGER
524,TXNX9004,70,13,2026-01-27 00:00:00,1499,Netbanking,captured,52,MISSING_IN_LEDGER
525,TXNX9005,252,27,2026-01-23 00:00:00,2999,Netbanking,captured,21,MISSING_IN_LEDGER
526,TXNX9006,43,37,2026-01-01 00:00:00,299,Card,captured,4,MISSING_IN_LEDGER
527,TXNX9007,141,15,2026-01-18 00:00:00,499,Wallet,captured,99,MISSING_IN_LEDGER
528,TXNX9008,235,37,2026-01-24 00:00:00,2999,UPI,captured,73,MISSING_IN_LEDGER
529,TXNX9009,59,18,2026-01-25 00:00:00,4999,Card,captured,69,MISSING_IN_LEDGER


In [4]:
##Amount and Status mismatches between Ledger and the Gateway report using Merged Dataframe##
common_ids=ledger_ids.intersection(gateway_ids)
ledger_common=ledger_df[ledger_df['transaction_id'].isin(common_ids)]
gateway_common=gateway_df[gateway_df['transaction_id'].isin(common_ids)]

print(f"Transactions present in both: {len(common_ids)}")
print("Common Transaction IDs in the Ledger- ")
display(ledger_common.head())

print(f"\nCommon Transaction IDs in the Gateway- ")
display(gateway_common.head())

merged_df=ledger_common.merge(gateway_common[['transaction_id','amount_inr','status']],
on='transaction_id',
suffixes=('_ledg','_gate'))

desired_columns=['transaction_id', 'user_id', 'merchant_id', 'transaction_time', 'amount_inr_ledg', 'amount_inr_gate', 'status_ledg', 'status_gate', 'payment_method', 'risk_score']
merged_df = merged_df[desired_columns]

print(f"\nMerged dataframe- ")
display(merged_df.head())

Transactions present in both: 520
Common Transaction IDs in the Ledger- 


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
1,TXN100001,287,27,2026-01-04 11:49:00,49,UPI,captured,79
2,TXN100002,122,11,2026-01-24 04:59:00,1499,Wallet,captured,22
3,TXN100003,92,22,2026-01-29 13:01:00,49,Netbanking,captured,100
4,TXN100004,343,16,2026-01-30 13:51:00,99,UPI,captured,34
6,TXN100006,236,23,2026-01-08 06:52:00,99,Wallet,captured,39



Common Transaction IDs in the Gateway- 


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
0,TXN100001,287,27,2026-01-04 11:49:00,49,UPI,captured,79
1,TXN100002,122,11,2026-01-24 04:59:00,1499,Wallet,captured,22
2,TXN100003,92,22,2026-01-29 13:01:00,49,Netbanking,captured,100
3,TXN100004,343,16,2026-01-30 13:51:00,99,UPI,captured,34
4,TXN100006,236,23,2026-01-08 06:52:00,99,Wallet,captured,39



Merged dataframe- 


,transaction_id,user_id,merchant_id,transaction_time,amount_inr_ledg,amount_inr_gate,status_ledg,status_gate,payment_method,risk_score
0,TXN100001,287,27,2026-01-04 11:49:00,49,49,captured,captured,UPI,79
1,TXN100002,122,11,2026-01-24 04:59:00,1499,1499,captured,captured,Wallet,22
2,TXN100003,92,22,2026-01-29 13:01:00,49,49,captured,captured,Netbanking,100
3,TXN100004,343,16,2026-01-30 13:51:00,99,99,captured,captured,UPI,34
4,TXN100006,236,23,2026-01-08 06:52:00,99,99,captured,captured,Wallet,39


In [10]:
##Amount Mismatches##
merged_df['amount_inr_diff']=merged_df['amount_inr_ledg']-merged_df['amount_inr_gate']
amount_mismatches=merged_df[merged_df['amount_inr_diff']!=0].copy()

print(f"Transactions with amount mismatch- {len(amount_mismatches)}")
if len(amount_mismatches)!=0:
    total_discrepancy=amount_mismatches['amount_inr_diff'].sum()
    print(f"\nTotal Discrepancy- ₹{total_discrepancy:,}")
    print("\nMismatch Details- ")
    print(amount_mismatches[['transaction_id', 'amount_inr_ledg', 'amount_inr_gate', 'amount_inr_diff']])

Transactions with amount mismatch- 16

Total Discrepancy- ₹-350

Mismatch Details- 
    transaction_id  amount_inr_ledg  amount_inr_gate  amount_inr_diff
9        TXN100011             2999             2949               50
17       TXN100019               49               99              -50
36       TXN100039             1499             1399              100
194      TXN100204               49              -51              100
207      TXN100218               49               -1               50
240      TXN100252              299              349              -50
257      TXN100270              799              899             -100
265      TXN100278              149               99               50
271      TXN100284             1499             1399              100
273      TXN100286               49              149             -100
287      TXN100301              149              249             -100
360      TXN100380              149              249             -100
363   

In [6]:
##Status Mismatches##
status_mismatches=merged_df[merged_df['status_ledg']!=merged_df['status_gate']].copy()

print(f"Transactions with status mismatch- {len(status_mismatches)}")
if len(status_mismatches)>0:
    print("\n⚠️  CRITICAL — STATUS ALTERED IN GATEWAY for following Transactions- \n")
    print(status_mismatches[['transaction_id', 'amount_inr_ledg', 'status_ledg', 'status_gate']])

Transactions with status mismatch- 9

⚠️  CRITICAL — STATUS ALTERED IN GATEWAY for following Transactions- 

    transaction_id  amount_inr_ledg status_ledg status_gate
39       TXN100042             1499    captured      failed
204      TXN100215              299    captured      failed
254      TXN100267              299    captured      failed
286      TXN100300               49    captured      failed
371      TXN100392               49    captured      failed
392      TXN100414             1499    captured      failed
463      TXN100487               49    captured      failed
482      TXN200008             2999  chargeback      failed
516      TXN300027              399    captured      failed


In [7]:
##---
##Part C Final Deliverable: Reusable Reconciliation Function
##The exploratory steps above have been wrapped into a single master function below to process the full datasets.

In [8]:
def reconcile_payments(ledger_df, gateway_df):
    """
    Compares internal ledger against external payment gateway to find discrepancies.
    Returns 4 DataFrames: Missing in Gateway, Missing in Ledger, Amount Mismatches, Status Mismatches.
    """
    # 1. SET OPERATIONS (Missing Transactions)
    ledger_ids=set(ledger_df['transaction_id'])
    gateway_ids=set(gateway_df['transaction_id'])
    
    missing_in_gateway = ledger_ids - gateway_ids
    missing_in_gateway_df = ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway)].copy()
    missing_in_gateway_df['issue_type'] = "MISSING_IN_GATEWAY"
    
    missing_in_ledger=gateway_ids - ledger_ids
    missing_in_ledger_df=gateway_df[gateway_df['transaction_id'].isin(missing_in_ledger)].copy()
    missing_in_ledger_df['issue_type'] = "MISSING_IN_LEDGER"
    
    # 2. MERGE OPERATIONS (Mismatched Data)
    common_ids=ledger_ids.intersection(gateway_ids)
    ledger_common=ledger_df[ledger_df['transaction_id'].isin(common_ids)]
    gateway_common=gateway_df[gateway_df['transaction_id'].isin(common_ids)]
    
    merged_df=ledger_common.merge(
        gateway_common[['transaction_id', 'amount_inr', 'status']],
        on='transaction_id',
        suffixes=('_ledg', '_gate')
    )
    
    # Amount mismatches
    merged_df['amount_inr_diff']=merged_df['amount_inr_ledg']-merged_df['amount_inr_gate']
    amount_mismatch_df=merged_df[merged_df['amount_inr_diff']!=0].copy()
    amount_mismatch_df['issue_type']="AMOUNT_MISMATCH"
    
    # Status mismatches
    status_mismatch_df=merged_df[merged_df['status_ledg']!=merged_df['status_gate']].copy()
    status_mismatch_df['issue_type']="STATUS_MISMATCH"
    
    # 3. RETURN ALL FOUR
    return missing_in_gateway_df, missing_in_ledger_df, amount_mismatch_df, status_mismatch_df

In [9]:
# Execute the final function
missing_gw, missing_ldg, amt_mismatch, stat_mismatch = reconcile_payments(ledger_df, gateway_df)

print("=========================================")
print("     PAYMENT RECONCILIATION REPORT       ")
print("=========================================")
print(f"Transactions Missing in Gateway: {len(missing_gw)} (~5%)")
print(f"Transactions Missing in Ledger:  {len(missing_ldg)} (~3%)")
print(f"Amount Discrepancies Found:      {len(amt_mismatch)} (~2%)")
print(f"Status Discrepancies Found:      {len(stat_mismatch)} (~2%)")
print("=========================================")

     PAYMENT RECONCILIATION REPORT       
Transactions Missing in Gateway: 27 (~5%)
Transactions Missing in Ledger:  10 (~3%)
Amount Discrepancies Found:      16 (~2%)
Status Discrepancies Found:      9 (~2%)
